In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

# 1. Load the Tokenizer and Model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 2. Prepare a tiny dataset
data = {
    "text": ["I love this!", "This is terrible", "Simply amazing", "I hate it"],
    "label": [1, 0, 1, 0] # 1 = Positive, 0 = Negative
}
ds = Dataset.from_dict(data)

# 3. Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_ds = ds.map(tokenize_function, batched=True)

# 4. Set Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds,
)

# 6. Train!
trainer.train()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

# 1. Load a real dataset (Rotten Tomatoes movie reviews)
dataset = load_dataset("rotten_tomatoes")

# 2. Load the "Tokenizer" (The bridge between text and IDs)
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 3. Process the text
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length")

tokenized_data = dataset.map(preprocess_function, batched=True)

# 4. Load the Pre-trained Brain
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

# 5. Define the training rules
training_args = TrainingArguments(
    output_dir="./movie_sentiment_model",
    evaluation_strategy="epoch", # Check accuracy after every round
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=1,          # Just 1 epoch for a quick demo
)

# 6. The Trainer (The engine)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
)

# 7. GO!
trainer.train()